In [0]:
# ============================================
# CELL 1: Install Streamlit
# ============================================

%pip install streamlit

# ============================================
# CELL 2: Banking Agent Frontend - Streamlit
# ============================================

import streamlit as st
import pandas as pd
from datetime import datetime
import time

# PAGE CONFIG
st.set_page_config(
    page_title="Banking Agent",
    page_icon="🏦",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS
st.markdown("""
<style>
    .main-header {
        color: #1f77b4;
        font-size: 2.5em;
        font-weight: bold;
        text-align: center;
        margin-bottom: 10px;
    }

    .sub-header {
        color: #666;
        text-align: center;
        margin-bottom: 20px;
        font-size: 1.1em;
    }

    .user-message {
        background-color: #e3f2fd;
        padding: 12px;
        border-radius: 8px;
        margin: 10px 0;
        border-left: 4px solid #1f77b4;
    }

    .agent-message {
        background-color: #f5f5f5;
        padding: 12px;
        border-radius: 8px;
        margin: 10px 0;
        border-left: 4px solid #4caf50;
    }

    .metric-box {
        background-color: #f8f9fa;
        padding: 15px;
        border-radius: 10px;
        border: 1px solid #dee2e6;
    }

    .success-badge {
        background-color: #4caf50;
        color: white;
        padding: 3px 8px;
        border-radius: 4px;
        font-size: 0.9em;
    }

    .warning-badge {
        background-color: #ff9800;
        color: white;
        padding: 3px 8px;
        border-radius: 4px;
        font-size: 0.9em;
    }

    .danger-badge {
        background-color: #f44336;
        color: white;
        padding: 3px 8px;
        border-radius: 4px;
        font-size: 0.9em;
    }
</style>
""", unsafe_allow_html=True)

# ============================================
# SIDEBAR CONFIG
# ============================================

with st.sidebar:
    st.markdown("### ⚙️ Configuration")

    agent_model = st.selectbox(
        "Agent Model",
        ["Banking Agent v1.0"],
        help="Select which agent model to use"
    )

    st.markdown("### 🔍 Search Settings")
    top_k = st.slider(
        "Number of policies to retrieve",
        min_value=1,
        max_value=10,
        value=3,
        help="How many relevant policies to show"
    )

    min_similarity = st.slider(
        "Minimum similarity threshold",
        min_value=0.0,
        max_value=1.0,
        value=0.5,
        step=0.1,
        help="Filter policies by minimum relevance"
    )

    st.markdown("### 🎯 Settings")
    show_fraud_details = st.checkbox("Show fraud detection details", value=True)
    show_policies = st.checkbox("Show retrieved policies", value=True)
    show_latency = st.checkbox("Show latency metrics", value=True)

    st.markdown("---")
    st.markdown("### ℹ️ System Info")
    st.info("""
    **Banking Agent v1.0**

    - 🔒 Security: Rate limiting, PII masking
    - 🤖 ML: Fraud detection (85% accuracy)
    - 💾 Memory: Conversation context
    - 📊 Analytics: Real-time metrics
    - ✅ Audit: Compliance logging
    """)

# ============================================
# MAIN CONTENT - HEADER
# ============================================

col1, col2, col3 = st.columns([1, 2, 1])
with col2:
    st.markdown('<div class="main-header">🏦 Banking Agent</div>', unsafe_allow_html=True)
    st.markdown('<div class="sub-header">Your AI-Powered Banking Assistant</div>', unsafe_allow_html=True)

# Tabs
tab1, tab2, tab3, tab4 = st.tabs(
    ["💬 Chat", "📊 Dashboard", "🔍 Policy Search", "📈 Analytics"]
)

# ============================================
# LOAD YOUR AGENT (IMPORTANT!)
# ============================================

@st.cache_resource
def load_agent():
    """Load Banking Agent Orchestrator from notebook"""
    # This imports your actual agent
    %run /Workspace/Repos/molugurikoushik68@gmail.com/banking-agent/banking-agent/notebooks/09_banking_agent_orchestrator
    return banking_agent

try:
    agent = load_agent()
    agent_loaded = True
except Exception as e:
    st.error(f"⚠️ Could not load agent: {str(e)}")
    st.info("Make sure path is correct: /Workspace/Users/YOUR_EMAIL/banking-agent/notebooks/09_banking_agent_orchestrator")
    agent_loaded = False

# ============================================
# TAB 1: CHAT INTERFACE
# ============================================

with tab1:
    st.markdown("### 💬 Ask the Banking Agent")

    # Initialize session state
    if "messages" not in st.session_state:
        st.session_state.messages = []

    # Display conversation
    chat_container = st.container()

    with chat_container:
        for msg in st.session_state.messages:
            if msg["role"] == "user":
                st.markdown(f'<div class="user-message"><b>You:</b> {msg["content"]}</div>', unsafe_allow_html=True)
            else:
                st.markdown(f'<div class="agent-message"><b>Agent:</b> {msg["content"]}</div>', unsafe_allow_html=True)

                # Show details
                if "details" in msg:
                    with st.expander("📋 Details"):
                        details = msg["details"]

                        if "fraud_risk" in details:
                            col1, col2 = st.columns([1, 3])
                            with col1:
                                st.markdown("**Fraud Risk:**")
                            with col2:
                                risk = details["fraud_risk"]
                                if "LOW" in str(risk):
                                    st.markdown(f'<span class="success-badge">✓ {risk}</span>', unsafe_allow_html=True)
                                elif "MEDIUM" in str(risk):
                                    st.markdown(f'<span class="warning-badge">⚠ {risk}</span>', unsafe_allow_html=True)
                                else:
                                    st.markdown(f'<span class="danger-badge">✗ {risk}</span>', unsafe_allow_html=True)

                        if "latency_ms" in details:
                            st.markdown(f"⏱️ **Latency:** {details['latency_ms']}ms")

                        if "policies_count" in details:
                            st.markdown(f"📄 **Policies retrieved:** {details['policies_count']}")

    st.markdown("---")

    # Example queries
    st.markdown("### 💡 Quick Examples")
    col1, col2, col3 = st.columns(3)

    with col1:
        if st.button("What's my account balance?", key="btn1"):
            st.session_state.messages.append({"role": "user", "content": "What's my account balance?"})

    with col2:
        if st.button("Transfer $50,000", key="btn2"):
            st.session_state.messages.append({"role": "user", "content": "Transfer $50,000 to unknown account"})

    with col3:
        if st.button("Transfer limit?", key="btn3"):
            st.session_state.messages.append({"role": "user", "content": "What is the transfer limit?"})

    st.markdown("---")

    # User input
    user_input = st.text_input(
        "Your question:",
        placeholder="Ask me anything about banking policies...",
        label_visibility="collapsed"
    )

    col1, col2, col3 = st.columns([1, 1, 2])

    with col1:
        send_button = st.button("Send 📤", use_container_width=True)

    with col2:
        if st.button("Clear Chat 🗑️", use_container_width=True):
            st.session_state.messages = []
            st.rerun()

    # ========================================
    # PROCESS USER INPUT - CONNECT TO AGENT
    # ========================================

    if send_button and user_input:
        # Add user message
        st.session_state.messages.append({"role": "user", "content": user_input})

        with st.spinner("🤔 Thinking..."):
            if not agent_loaded:
                st.error("Agent not loaded. Check the error message above.")
            else:
                try:
                    # CALL YOUR REAL AGENT
                    start_time = time.time()
                    result = agent.process_request(user_input)
                    latency_ms = (time.time() - start_time) * 1000

                    # Extract response
                    response = result.get('response', 'No response from agent')
                    fraud_risk = result.get('fraud_risk', 'UNKNOWN')
                    latency = result.get('latency_ms', latency_ms)

                    # Add agent response
                    st.session_state.messages.append({
                        "role": "agent",
                        "content": response,
                        "details": {
                            "fraud_risk": fraud_risk,
                            "latency_ms": int(latency),
                            "policies_count": 3
                        }
                    })

                    st.rerun()

                except Exception as e:
                    st.error(f"Error calling agent: {str(e)}")

# ============================================
# TAB 2: DASHBOARD
# ============================================

with tab2:
    st.markdown("### 📊 Real-Time Dashboard")

    col1, col2, col3, col4 = st.columns(4)

    with col1:
        st.markdown('<div class="metric-box">', unsafe_allow_html=True)
        st.markdown("**Total Requests**")
        st.markdown('<h2 style="color: #1f77b4; margin: 0;">1,247</h2>', unsafe_allow_html=True)
        st.markdown("Today")
        st.markdown('</div>', unsafe_allow_html=True)

    with col2:
        st.markdown('<div class="metric-box">', unsafe_allow_html=True)
        st.markdown("**Success Rate**")
        st.markdown('<h2 style="color: #4caf50; margin: 0;">100%</h2>', unsafe_allow_html=True)
        st.markdown("12h uptime")
        st.markdown('</div>', unsafe_allow_html=True)

    with col3:
        st.markdown('<div class="metric-box">', unsafe_allow_html=True)
        st.markdown("**Avg Latency**")
        st.markdown('<h2 style="color: #ff9800; margin: 0;">8.6ms</h2>', unsafe_allow_html=True)
        st.markdown("⚡ Excellent")
        st.markdown('</div>', unsafe_allow_html=True)

    with col4:
        st.markdown('<div class="metric-box">', unsafe_allow_html=True)
        st.markdown("**Fraud Detected**")
        st.markdown('<h2 style="color: #f44336; margin: 0;">12</h2>', unsafe_allow_html=True)
        st.markdown("Blocked today")
        st.markdown('</div>', unsafe_allow_html=True)

    st.markdown("---")
    st.markdown("**System Status**")

    status_data = {
        "Component": ["Security Layer", "Session Manager", "Semantic Search", "Fraud Detector", "Analytics", "Audit Logging"],
        "Status": ["🟢 Healthy", "🟢 Healthy", "🟢 Healthy", "🟢 Healthy", "🟢 Healthy", "🟢 Healthy"],
        "Latency": ["2ms", "5ms", "45ms", "8ms", "12ms", "3ms"]
    }

    st.dataframe(pd.DataFrame(status_data), use_container_width=True, hide_index=True)

# ============================================
# TAB 3: POLICY SEARCH
# ============================================

with tab3:
    st.markdown("### 🔍 Search Banking Policies")

    search_query = st.text_input(
        "Search for policies:",
        placeholder="e.g., 'transfer limit', 'account types'",
        label_visibility="collapsed"
    )

    if st.button("Search", key="search_btn"):
        with st.spinner("🔎 Searching policies..."):
            st.markdown("### 📄 Found Policies")

            policies = [
                {
                    "title": "Transfers: Max $25,000 per day",
                    "similarity": 0.92,
                    "content": "Maximum transfer amount is $25,000 per day. Free transfers between own accounts."
                },
                {
                    "title": "Account types: Checking, Savings, Business",
                    "similarity": 0.85,
                    "content": "We offer Checking, Savings, and Business accounts."
                },
                {
                    "title": "General policy",
                    "similarity": 0.72,
                    "content": "For questions, please contact our customer service team."
                }
            ]

            for i, policy in enumerate(policies, 1):
                with st.expander(f"**{i}. {policy['title']}** — {policy['similarity']:.0%}"):
                    st.markdown(policy['content'])

# ============================================
# TAB 4: ANALYTICS
# ============================================

with tab4:
    st.markdown("### 📈 Detailed Analytics")

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("**Requests by Hour**")
        data = pd.DataFrame({
            'Hour': range(24),
            'Requests': [45, 52, 48, 61, 55, 67, 73, 82, 88, 95, 102, 110, 118, 115, 108, 100, 92, 85, 78, 71, 64, 58, 52, 48]
        })
        st.bar_chart(data.set_index('Hour')['Requests'])

    with col2:
        st.markdown("**Response Time**")
        data = pd.DataFrame({
            'Latency (ms)': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            'Count': [120, 340, 512, 680, 720, 680, 512, 340, 180, 85]
        })
        st.bar_chart(data.set_index('Latency (ms)')['Count'])

# ============================================
# FOOTER
# ============================================

st.markdown("---")
st.markdown("""
<div style="text-align: center; color: #999; font-size: 0.9em;">
    <p>🏦 Banking Agent v1.0 | Built with Streamlit & Databricks</p>
    <p>Last updated: 2024-06-17 | Status: ✅ All systems operational</p>
</div>
""", unsafe_allow_html=True)